# N082 · 拓扑排序与依赖图

**目标：** 判断是否存在合法依赖顺序并处理多解。

**先修：** N079, N080, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 入度；Kahn；三色DFS；DAG；环；偏序与全序。

## 从问题到算法

依赖边u→v表示u必须先完成。入度0顶点没有尚未满足的前置条件，可以安全输出；删除它的出边后可能解锁新顶点。若最终仍有未输出顶点，则剩余子图每点都有前驱，沿前驱回溯必形成环。拓扑次序可能不唯一，测试检查每条边顺序而不是强行匹配一份列表。

## 最小实现

**本章接口：** `topological_sort(n, edges)`、`can_finish(n, prerequisites)`、`find_order(n, prerequisites)`

In [1]:
from collections import deque

def topological_sort(n,edges):
    adj=[[] for _ in range(n)]; indegree=[0]*n
    for u,v in edges: adj[u].append(v); indegree[v]+=1
    queue=deque(i for i in range(n) if indegree[i]==0); order=[]
    while queue:
        u=queue.popleft(); order.append(u)
        for v in adj[u]:
            indegree[v]-=1
            if indegree[v]==0: queue.append(v)
    return order if len(order)==n else None

def can_finish(n,prerequisites):
    return topological_sort(n,[(required,course) for course,required in prerequisites]) is not None

def find_order(n,prerequisites):
    order=topological_sort(n,[(required,course) for course,required in prerequisites])
    return [] if order is None else order

## 正确性、前提与复杂度

当前入度精确等于来自未输出顶点的前驱数。每次取零入度点不会违反任何边；无法继续且未结束则证明存在有向环。

**代价：** O(V+E)时间与空间。prerequisites使用[course,required]输入顺序，构图时需反转。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(0,2),(1,3),(2,3)]; order=topological_sort(4,edges)
show_table(['位置','顶点','必须在它前面的顶点'],[(i,v,[u for u,w in edges if w==v]) for i,v in enumerate(order)])

位置,顶点,必须在它前面的顶点
0,0,[]
1,1,[0]
2,2,[0]
3,3,"[1, 2]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert can_finish(2,[[1,0]]) and not can_finish(2,[[1,0],[0,1]])
assert topological_sort(0,[])==[] and topological_sort(1,[(0,0)]) is None
from random import Random
rng=Random(82)
for n in range(1,25):
    permutation=list(range(n)); rng.shuffle(permutation)
    edges=[(permutation[i],permutation[j]) for i in range(n) for j in range(i+1,n) if rng.randrange(5)==0]
    order=topological_sort(n,edges); position={x:i for i,x in enumerate(order)}
    assert len(order)==n and all(position[u]<position[v] for u,v in edges)
assert set(find_order(4,[]))==set(range(4))
print('N082: 所有本章断言通过')

N082: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 返回任一合法顺序而不是指定顺序。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 扩展输出每门课最早层次。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 207. Course Schedule（canonical）：[官方入口](https://leetcode.com/problems/course-schedule/)
- 210. Course Schedule II（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。